<div id="singlestore-header" style="display: flex; background-color: rgba(209, 153, 255, 0.25); padding: 5px;">
    <div id="icon-image" style="width: 90px; height: 90px;">
        <img width="100%" height="100%" src="https://raw.githubusercontent.com/singlestore-labs/spaces-notebooks/master/common/images/header-icons/vector-circle.png" />
    </div>
    <div id="text" style="padding: 5px; margin-left: 10px;">
        <div id="badge" style="display: inline-block; background-color: rgba(0, 0, 0, 0.15); border-radius: 4px; padding: 4px 8px; align-items: center; margin-top: 6px; margin-bottom: -2px; font-size: 80%">SingleStore Notebooks</div>
        <h1 style="font-weight: 500; margin: 8px 0 0 4px;">Build Modern AI and Search Applications</h1>
    </div>
</div>

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Note</b></p>
        <p>This notebook can be run on a Free Starter Workspace. To create a Free Starter Workspace navigate to <tt>Start</tt> using the left nav. You can also use your existing Standard or Premium workspace with this Notebook.</p>
    </div>
</div>

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Note</b></p>
        <p>F16 vectors, the full-text index merger, <tt>SHOW FULLTEXT INDEX STATUS</tt> and <tt>ANALYZE FULLTEXT</tt> need a recent SingleStore version (9.1 or later on self-managed). The notebook checks your version first and skips anything your workspace doesn't support, with a clear message instead of an error.</p>
        <p>Vectors are created in SQL with SingleStore's AI function <tt>cluster.EMBED_TEXT</tt>. If AI Functions aren't enabled on your workspace (AI &gt; AI &amp; ML Functions in the portal), the notebook falls back to a free open-source embedding model, so it runs either way.</p>
    </div>
</div>

**Demonstrate F16 vectors, full-text search, index merging, and improved full-text index manageability.**

## The story

Congratulations. You just joined **AdVenture Inc.**, a real-time ad startup, as its only developer. Your platform serves location-based offers to shoppers in New York using the SingleStore **Real-Time Digital Marketing** dataset: advertisers, their offers, and the audience segments they target.

Your new boss is **Morgan Blaze**, Chief Marketing Officer. Morgan is brilliant, loud, and sends requests to `#war-room` at all hours. Each request in this notebook unlocks a feature:

| Morgan's request | What you build | SingleStore feature |
| --- | --- | --- |
| "Turn these boring offers into real ads" | A pipeline that streams purchases and creates embedded ads as it loads | Pipelines + AI Functions (`cluster.EMBED_TEXT`) |
| "Why can't I just search for 'wireless'?" | Keyword search that forgives typos | Full-text search VERSION 2, BM25 |
| "The search thinks 'running' and 'run' are different words" | Swap the text analyzer without downtime | Full-text index manageability: `ANALYZE FULLTEXT`, `SHOW FULLTEXT INDEX STATUS` |
| "Find ads that *feel like a rainy Sunday*" | Semantic search | F16 vectors + vector index |
| "Give me both. Now." | Hybrid search | Reciprocal Rank Fusion in one SQL query |
| "Super Bowl traffic is 10x. Don't you dare slow down." | Stream new ads, then merge each index type | Vector index merging and full-text index merging (two different mechanisms) |

**Who this is for:** developers trying SingleStore (every step is copy-paste friendly), plus **Evaluator's corner** boxes with measurements for tech leads comparing databases.

## Chapter 0: Your first day

Install the helper libraries. `sentence-transformers` is only used if AI Functions aren't available on your workspace.

In [1]:
%pip install singlestoredb pandas --quiet

In [2]:
import json
import random
import re
import statistics
import time

import pandas as pd
import singlestoredb as s2

pd.set_option('display.max_colwidth', 120)
random.seed(42)

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Action Required</b></p>
        <p>If you have a Free Starter Workspace deployed already, select its database from the drop-down menu at the top of this notebook. On a Standard or Premium workspace, the next cell creates a database called <tt>adventure_inc</tt> for you.</p>
    </div>
</div>

In [3]:
import json, random, re, statistics, time   # repeated here so this cell works on its own
import pandas as pd
import singlestoredb as s2

DB_NAME = 'adventure_inc'

USE_DB = None   # set below once we know which database to use

def connect():
    """Open a connection (again) and select the notebook's database."""
    global conn
    conn = s2.connect()
    conn.autocommit(True)
    if USE_DB:
        with conn.cursor() as cur:
            cur.execute(f'USE {USE_DB}')

def _query(query, params=None):
    with conn.cursor() as cur:
        cur.execute(query, params)
        if cur.description:
            cols = [d[0] for d in cur.description]
            return pd.DataFrame(cur.fetchall(), columns=cols)
    return None

def run_sql(query, params=None):
    """Run a query and return a DataFrame (or None). Reconnects once if an idle connection was dropped."""
    try:
        return _query(query, params)
    except Exception as e:
        if getattr(e, 'errno', None) in (2006, 2013) or 'Lost connection' in str(e) or 'gone away' in str(e):
            print('(Connection was dropped, reconnecting...)')
            connect()
            return _query(query, params)
        raise

connect()

def try_sql(query, params=None, label='This step'):
    """Like run_sql(), but explains and continues if the workspace doesn't support something."""
    try:
        return run_sql(query, params)
    except Exception as e:
        print(f'{label} was skipped: {e}')
        return None

shared = run_sql("SHOW VARIABLES LIKE 'is_shared_tier'")
IS_SHARED = shared is not None and len(shared) > 0 and shared.iloc[0, 1] == 'ON'
if not IS_SHARED:
    run_sql(f'CREATE DATABASE IF NOT EXISTS {DB_NAME}')
    USE_DB = DB_NAME
    run_sql(f'USE {DB_NAME}')

print('Connected to database:', run_sql('SELECT DATABASE() AS db').iloc[0, 0])
print('SingleStore version:  ', run_sql('SELECT @@memsql_version AS v').iloc[0, 0])

### Feature check

Morgan doesn't care about version numbers, but you do. These checks decide which path the notebook takes.

In [4]:
import logging, os, warnings
os.environ['HF_HUB_VERBOSITY'] = 'error'              # quiet the model download messages
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
for name in ['httpx', 'huggingface_hub', 'huggingface_hub.utils._http', 'sentence_transformers']:
    logging.getLogger(name).setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message='.*unauthenticated requests.*')

# 1. F16 vectors (half precision)
try:
    run_sql("SELECT '[0.5, -0.25]' :> VECTOR(2, F16) AS v")
    VEC_ELEM = 'F16'
    print('F16 vectors: supported')
except Exception as e:
    VEC_ELEM = 'F32'
    print('F16 vectors: not available on this workspace, falling back to F32 ->', e)

# 2. AI Functions for embeddings
local_model = None
try:
    probe = run_sql("SELECT LENGTH(cluster.EMBED_TEXT('hello from AdVenture Inc.')) AS n_bytes")
    DIM = int(probe.iloc[0, 0]) // 4   # EMBED_TEXT returns packed 32-bit floats
    run_sql(f"SELECT cluster.EMBED_TEXT('check') :> VECTOR({DIM}) :> VECTOR({DIM}, {VEC_ELEM}) AS v")
    EMBED_MODE = 'ai'
    print(f'AI Functions: available, embeddings have {DIM} dimensions')
except Exception as e:
    EMBED_MODE = 'local'
    print('AI Functions: not available, using a free open-source model instead ->', e)
    print('Downloading the model from Hugging Face (one time, about a minute)...')
    get_ipython().run_line_magic('pip', 'install sentence-transformers --quiet')
    from sentence_transformers import SentenceTransformer
    local_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    DIM = (getattr(local_model, 'get_embedding_dimension', None)
           or local_model.get_sentence_embedding_dimension)()

VEC_TYPE = f'VECTOR({DIM}, {VEC_ELEM})'
print('Embedding column type:', VEC_TYPE, '| embeddings made by:', EMBED_MODE)

## Chapter 1: Boring offers in, real ads out

<div style="border-left: 4px solid #d199ff; background: rgba(209,153,255,0.10); padding: 10px 14px; margin: 8px 0;">
<b>#war-room</b> &nbsp;<span style="opacity:0.7">8:02 AM</span><br/>
<b>Morgan Blaze, Chief Marketing Officer:</b> Welcome aboard! Quick thing. Our offers say stuff like <i>"37% off at Wordtune"</i>. That's not an ad, that's a receipt. I want real ad copy for every offer, and I want it searchable by lunch.
</div>

The Real-Time Digital Marketing dataset lives in a public S3 bucket. It streams **purchase events** from simulated shoppers in New York (who bought from which vendor), and comes with a list of about 1,000 **vendors** and their categories.

Offers in the original demo app are generated from that vendor list, and so are ours: `"<10 to 50>% off at <vendor>"`, a target domain and a bid. That's short text, so we'll enrich it as it loads. The **pipeline itself** does the heavy lifting: as purchases stream in, any vendor seen for the first time gets an offer, a headline and ad copy based on its category, and a vector from `cluster.EMBED_TEXT`, all in the same step. That gives the full-text index plenty of text, and every ad arrives search-ready.

First, two small lookup tables: the dataset's vendor list and a "vibe book" your copywriter left behind.

In [5]:
vendors = pd.read_json('https://raw.githubusercontent.com/singlestore-labs/demo-realtime-digital-marketing/main/data/vendors.json')
vendors['domain'] = (vendors['vendor'].str.lower() + '.' + vendors['tld'])
vendors = vendors.drop_duplicates('domain')

run_sql('DROP TABLE IF EXISTS vendors')
run_sql('''
    CREATE ROWSTORE REFERENCE TABLE vendors (
        domain VARCHAR(200) PRIMARY KEY,
        vendor VARCHAR(100),
        category VARCHAR(50),
        popularity DOUBLE
    )
''')
with conn.cursor() as cur:
    cur.executemany(
        'INSERT INTO vendors VALUES (%s, %s, %s, %s)',
        vendors[['domain', 'vendor', 'category', 'popularity']].values.tolist(),
    )
run_sql('SELECT category, COUNT(*) AS vendors FROM vendors GROUP BY category ORDER BY vendors DESC LIMIT 8')

In [6]:
# The copywriter's vibe book: three hooks, a story and a call to action per category.
VIBES = {
    'books':       ('Plot twist: you finally read it', 'Your reading nook called', 'Chapter one starts tonight',
                    'Rain on the window, tea in hand, a stack of new stories and nowhere to be on a slow Sunday.', 'Curl up and start reading.'),
    'games':       ('Press start on the weekend', 'Couch co-op, snacks ready', 'One more level, we promise',
                    'Stay-in weekends, board game nights with friends and cozy gaming marathons when the weather turns grey.', 'Grab a controller.'),
    'movies':      ('Popcorn is mandatory', 'Your movie night just got an upgrade', 'Roll credits on boredom',
                    'Blanket forts, film marathons and date nights on the sofa while it pours outside.', 'Hit play.'),
    'music':       ('Turn it up', 'Your road trip needs a soundtrack', 'Headphones on, world off',
                    'Wireless headphones, vinyl crates and playlists for long drives, workouts and quiet rainy afternoons.', 'Find your sound.'),
    'electronics': ('Upgrade everything', 'Wireless and fearless', 'Charged up and ready',
                    'Wireless headphones, smart speakers and gadgets that make your home, commute and workouts easier.', 'Plug into the future.'),
    'computers':   ('Built for deadline day', 'Your laptop deserves better', 'Faster than your group chat',
                    'Laptops, monitors and keyboards for working from home, studying late and gaming after hours.', 'Power up your desk.'),
    'sports':      ('Game day ready', 'Run further, recover faster', 'Your personal best is waiting',
                    'Running shoes, team jerseys and gear for runners, weekend leagues and big game watch parties.', 'Get in the game.'),
    'outdoors':    ('Adventure is calling', 'Tents up, phones down', 'Trail mix not included',
                    'Camping gear, hiking boots and everything for a road trip into the mountains or a lazy lake weekend.', 'Head outside.'),
    'shoes':       ('Step it up', 'Blisters are so last season', 'Run the city in comfort',
                    'Running shoes, sneakers and boots made for long walks, city commutes and marathon training.', 'Find your fit.'),
    'clothing':    ('New season, new you', 'Your closet called, it is bored', 'Layer up in style',
                    'Cozy sweaters for rainy days, outfits for date night and everyday basics that just work.', 'Dress the part.'),
    'beauty':      ('Glow like you mean it', 'Self-care Sunday, sorted', 'Your skin says thank you',
                    'Skincare, candles and bath bombs for a relaxing spa day at home or getting ready for a night out.', 'Treat yourself.'),
    'jewelry':     ('Say it with sparkle', 'Anniversary panic? Solved', 'Shine on',
                    'Rings, necklaces and gifts for anniversaries, date nights, proposals and holiday surprises.', 'Make it unforgettable.'),
    'health':      ('Feel good from the inside', 'Wellness, minus the lecture', 'Your body will notice',
                    'Vitamins, yoga mats and sleep aids for calm mornings, healthy habits and a fresh start.', 'Start feeling better.'),
    'home':        ('Make your place a sanctuary', 'Cozy mode: activated', 'Home is where the throw pillows are',
                    'Blankets, candles and furniture that turn a grey rainy day into the coziest day of the year.', 'Nest a little.'),
    'garden':      ('Grow something good', 'Green thumbs welcome', 'Your balcony, but lush',
                    'Seeds, planters and tools for spring planting, herb gardens and slow weekends outdoors.', 'Get your hands dirty.'),
    'grocery':     ('Dinner, solved', 'Snack attack approved', 'Fresh from the market',
                    'Fresh groceries, comfort food ingredients and snacks for movie nights and busy weeknights.', 'Fill your cart.'),
    'kids':        ('Keep the little ones busy', 'Rainy day rescue kit', 'Fun for the whole family',
                    'Crafts, puzzles and activities for rainy days inside, school breaks and family weekends.', 'Make some memories.'),
    'toys':        ('Playtime, upgraded', 'The favorite toy is here', 'Bored? Not anymore',
                    'Building sets, puzzles and games for birthdays, holidays and rainy afternoons at home.', 'Let the fun begin.'),
    'baby':        ('Tiny humans, big deals', 'Sleep more (maybe)', 'Everything for the new arrival',
                    'Soft blankets, strollers and nursery essentials for new parents and baby showers.', 'Welcome the little one.'),
    'automotive':  ('Road trip ready', 'Your car deserves a spa day', 'Built for the long haul',
                    'Car care, tires and accessories for road trips, daily commutes and winter driving.', 'Hit the road.'),
    'tools':       ('Weekend project? Nailed it', 'Fix it yourself', 'Measure twice, save once',
                    'Drills, toolkits and supplies for DIY weekends, home repairs and garage projects.', 'Build something.'),
    'industrial':  ('Heavy duty, light price', 'Built for the job site', 'Tough gear for tough days',
                    'Safety gear, equipment and supplies for warehouses, workshops and serious builders.', 'Get to work.'),
    'general':     ('Deals worth stopping for', 'You will want to see this', 'Something good nearby',
                    'Everyday deals from local shops around the city.', 'Take a look.'),
}

run_sql('DROP TABLE IF EXISTS category_vibes')
run_sql('''
    CREATE ROWSTORE REFERENCE TABLE category_vibes (
        category VARCHAR(50) PRIMARY KEY,
        hook1 TEXT, hook2 TEXT, hook3 TEXT, story TEXT, cta TEXT
    )
''')
with conn.cursor() as cur:
    cur.executemany('INSERT INTO category_vibes VALUES (%s, %s, %s, %s, %s, %s)',
                    [(k, *v) for k, v in VIBES.items()])
print(f'{len(VIBES)} categories in the vibe book')

### The tables

- `purchases`: the live purchase stream from S3.
- `offers`: one offer per advertiser domain, created the first time that vendor shows up in the stream.
- `ad_creatives`: generated ad copy plus an embedding column of type `VECTOR(<dims>, F16)`.

<div class="alert alert-block alert-info"><b class="fa fa-solid fa-lightbulb"></b><div><p><b>Why F16?</b> Half-precision vectors use about half the storage of F32 and, per SingleStore's docs, give about 35% faster exact kNN search with similar speed and recall for approximate (indexed) search. DOT_PRODUCT and EUCLIDEAN_DISTANCE work directly on F16. If F16 isn't available on your workspace yet, the notebook uses F32 and everything else works the same.</p></div></div>

Notice there is **no full-text index yet**. We "forgot" it on purpose. You'll add it later without rebuilding anything.

In [7]:
MAX_CREATIVES = 3000   # caps AI Function calls per batch, keep it modest on a trial
SCALE = '100k-4p'      # dataset size in the bucket: 1k-2p, 10k-2p, 100k-4p, 1m-16p, ... 10m-80p

try_sql('DROP PIPELINE IF EXISTS purchases_pipeline', label='Dropping an old pipeline')
for t in ['purchases', 'offers', 'ad_creatives', 'creatives_staging']:
    run_sql(f'DROP TABLE IF EXISTS {t}')

run_sql('''
    CREATE TABLE purchases (
        subscriber_id BIGINT,
        vendor VARCHAR(100),
        ts DATETIME(6) DEFAULT NOW(6),
        SORT KEY (ts),
        KEY (vendor) USING HASH
    )
''')

run_sql('''
    CREATE ROWSTORE TABLE offers (
        offer_id BIGINT PRIMARY KEY,
        customer VARCHAR(100),
        enabled BOOL,
        notification_zone TEXT,
        segment_ids JSON,
        notification_content TEXT,
        notification_target VARCHAR(200),
        maximum_bid_cents BIGINT
    )
''')

run_sql('''
    CREATE ROWSTORE TABLE creatives_staging (
        offer_id BIGINT PRIMARY KEY,
        customer VARCHAR(100), category VARCHAR(50), domain VARCHAR(200),
        headline TEXT, body TEXT, max_bid_cents BIGINT
    )
''')

run_sql(f'''
    CREATE TABLE ad_creatives (
        offer_id BIGINT NOT NULL,
        customer VARCHAR(100),
        category VARCHAR(50),
        domain VARCHAR(200),
        headline TEXT,
        body TEXT,
        max_bid_cents BIGINT,
        embedding {VEC_TYPE},
        created_at DATETIME(6) DEFAULT NOW(6),
        SORT KEY (created_at),
        SHARD KEY (offer_id)
    )
''')
print('Tables ready.')

### The creative factory

`build_creatives` turns new offers into ads in two steps:

1. Write the copy into a small staging table (plain SQL: `ELT` picks one of three hooks per offer).
2. Move the staged rows into `ad_creatives`, calling `cluster.EMBED_TEXT` on each ad **inside the same `INSERT ... SELECT`**.

Staging first means the AI function only ever sees brand-new rows, so you never pay to re-embed old ones.

In [8]:
# Embed the ad's meaning, not the made-up brand names: strip the vendor name and domain first.
embed_text_sql = "REPLACE(REPLACE(CONCAT(s.headline, '. ', s.body), s.domain, ''), s.customer, '')"
embed_expr = (f"cluster.EMBED_TEXT({embed_text_sql}) :> VECTOR({DIM}) :> {VEC_TYPE}"
              if EMBED_MODE == 'ai' else 'NULL')

run_sql(f'''
CREATE OR REPLACE PROCEDURE build_creatives() AS
BEGIN
    INSERT INTO creatives_staging
    SELECT o.offer_id,
           o.customer,
           COALESCE(v.category, 'general'),
           o.notification_target,
           CONCAT(o.notification_content, ': ', ELT(1 + MOD(o.offer_id, 3), cv.hook1, cv.hook2, cv.hook3)),
           CONCAT(cv.story, ' ', o.customer, ' (', o.notification_target, ') bids up to ',
                  o.maximum_bid_cents, ' cents to reach you. ', cv.cta),
           o.maximum_bid_cents
    FROM offers o
    LEFT JOIN vendors v ON v.domain = o.notification_target
    JOIN category_vibes cv ON cv.category = COALESCE(v.category, 'general')
    WHERE NOT EXISTS (SELECT 1 FROM ad_creatives a WHERE a.offer_id = o.offer_id)
    LIMIT {MAX_CREATIVES};

    INSERT INTO ad_creatives (offer_id, customer, category, domain, headline, body, max_bid_cents, embedding)
    SELECT s.offer_id, s.customer, s.category, s.domain, s.headline, s.body, s.max_bid_cents,
           {embed_expr}
    FROM creatives_staging s;

    DELETE FROM creatives_staging;
END
''')

run_sql('''
CREATE OR REPLACE PROCEDURE refresh_ads() AS
BEGIN
    -- One offer per advertiser domain, generated the same way as the original demo app.
    INSERT IGNORE INTO offers
    SELECT CRC32(v.domain),
           v.vendor,
           TRUE,
           NULL,
           '[]',
           CONCAT(10 + MOD(CRC32(v.domain), 41), '% off at ', v.vendor),
           v.domain,
           2 + MOD(CRC32(v.domain), 14)
    FROM vendors v
    WHERE v.vendor IN (SELECT DISTINCT vendor FROM purchases);

    CALL build_creatives();
END
''')

run_sql('''
CREATE OR REPLACE PROCEDURE ingest_purchases(batch QUERY(subscriber_id BIGINT, vendor VARCHAR(100))) AS
BEGIN
    INSERT INTO purchases (subscriber_id, vendor) SELECT subscriber_id, vendor FROM batch;
    CALL refresh_ads();
END
''')

def embed_pending():
    """Local-model fallback: fill any NULL embeddings from Python."""
    if EMBED_MODE != 'local':
        return
    todo = run_sql('SELECT offer_id, customer, domain, headline, body FROM ad_creatives WHERE embedding IS NULL')
    if todo is None or todo.empty:
        return
    texts = [(h + '. ' + b).replace(d, '').replace(c, '')      # same brand-name stripping as the SQL path
             for h, b, d, c in zip(todo['headline'], todo['body'], todo['domain'], todo['customer'])]
    vecs = local_model.encode(texts, normalize_embeddings=True)
    with conn.cursor() as cur:
        cur.executemany('UPDATE ad_creatives SET embedding = %s WHERE offer_id = %s',
                        [(json.dumps([float(x) for x in v]), int(i)) for v, i in zip(vecs, todo['offer_id'])])
    print(f'Embedded {len(todo)} ads locally.')

print('Procedures ready.')

### Start the pipeline

The pipeline reads the purchase files for the chosen `SCALE` from the public `singlestore-realtime-digital-marketing` bucket and hands each batch to `ingest_purchases`, which stores the purchases and creates ads (with vectors) for any new advertisers. Running it in the `FOREGROUND` means the cell waits until the load is done. In production you'd leave it running in the background so new events stream in continuously.

In [9]:
BUCKET_URL = 'https://singlestore-realtime-digital-marketing.s3.amazonaws.com'

try:
    run_sql(f'''
        CREATE OR REPLACE PIPELINE purchases_pipeline AS
        LOAD DATA S3 'singlestore-realtime-digital-marketing/v2/{SCALE}/purchases.*'
        CREDENTIALS '{{}}'
        CONFIG '{{"region": "us-east-1"}}'
        INTO PROCEDURE ingest_purchases
        FORMAT PARQUET (
            subscriber_id <- subscriberid,
            vendor <- vendor
        )
    ''')
    t0 = time.perf_counter()
    run_sql('START PIPELINE purchases_pipeline FOREGROUND')
    print(f'Pipeline finished in {time.perf_counter() - t0:.1f}s')
except Exception as e:
    # Fallback: read a few of the same files over HTTPS and run the same procedure logic.
    import io, urllib.request
    print('Pipeline not available here, loading over HTTPS instead ->', e)
    listing = urllib.request.urlopen(f'{BUCKET_URL}/?list-type=2&prefix=v2/{SCALE}/purchases.').read().decode()
    for key in re.findall(r'<Key>(.*?)</Key>', listing)[:10]:
        df = pd.read_parquet(io.BytesIO(urllib.request.urlopen(f'{BUCKET_URL}/{key}').read()))
        with conn.cursor() as cur:
            cur.executemany('INSERT INTO purchases (subscriber_id, vendor) VALUES (%s, %s)',
                            df[['subscriberid', 'vendor']].values.tolist())
    run_sql('CALL refresh_ads()')

embed_pending()
run_sql('''
    SELECT (SELECT COUNT(*) FROM purchases)    AS purchases_loaded,
           (SELECT COUNT(*) FROM offers)       AS offers_created,
           (SELECT COUNT(*) FROM ad_creatives) AS ads_created
''')

In [10]:
run_sql('''
    SELECT offer_id, customer, category, headline, body
    FROM ad_creatives
    ORDER BY RAND()
    LIMIT 5
''')

Because the purchases are real events, you can already see which advertisers are hot right now:

In [11]:
run_sql('''
    SELECT p.vendor, p.purchases, a.category, a.headline
    FROM (SELECT vendor, COUNT(*) AS purchases FROM purchases
          GROUP BY vendor ORDER BY purchases DESC LIMIT 5) p
    JOIN (SELECT customer, ANY_VALUE(category) AS category, ANY_VALUE(headline) AS headline
          FROM ad_creatives GROUP BY customer) a ON a.customer = p.vendor
    ORDER BY p.purchases DESC
''')

<div style="border-left: 4px solid #d199ff; background: rgba(209,153,255,0.10); padding: 10px 14px; margin: 8px 0;">
<b>#war-room</b> &nbsp;<span style="opacity:0.7">11:48 AM</span><br/>
<b>Morgan Blaze, Chief Marketing Officer:</b> OK these are actual ads now. Emotional. Beautiful. Now I want to SEARCH them.
</div>

## Chapter 2: "Why can't I just search for 'wireless'?"

Time to add the full-text index you "forgot". With VERSION 2 full-text search you can add one to a table that already has data:

In [12]:
run_sql('''
    ALTER TABLE ad_creatives
    ADD FULLTEXT USING VERSION 2 ft_ads (customer, headline, body)
    INDEX_OPTIONS '{"analyzer": "standard"}'
''')
run_sql('OPTIMIZE TABLE ad_creatives FLUSH')   # make the newest rows searchable right away
print('Full-text index added.')

<div class="alert alert-block alert-info"><b class="fa fa-solid fa-gears"></b><div>
<p><b>Under the hood</b></p>
<p>VERSION 2 full-text search is built on <b>Apache Lucene</b>, the open-source Java search library that Elasticsearch, OpenSearch and Solr are also built on. SingleStore runs Lucene in a companion Java process it calls the <b>JLucene service</b>, on the same machine as the engine, talking to it over domain sockets and shared memory. Lucene builds one index per columnstore segment, which matters in Chapter 6.</p>
</div></div>

### Improved manageability: is the index ready?

Adding an index to existing data happens in the background. `SHOW FULLTEXT INDEX STATUS` reports progress per partition, so you can answer Morgan's "is it done yet?" with facts:

In [13]:
try_sql('SHOW FULLTEXT INDEX STATUS FOR ad_creatives', label='SHOW FULLTEXT INDEX STATUS')

If `Broken (%)` ever shows anything above zero, `OPTIMIZE TABLE ad_creatives FIX_FULLTEXT` repairs it.

### First searches

VERSION 2 uses Lucene-style queries: `column:term`, phrases in quotes, `AND` / `OR`, and `~` for fuzzy matching. `BM25` scores how well each ad matches.

<div style="border-left: 4px solid #d199ff; background: rgba(209,153,255,0.10); padding: 10px 14px; margin: 8px 0;">
<b>#war-room</b> &nbsp;<span style="opacity:0.7">12:03 PM</span><br/>
<b>Morgan Blaze, Chief Marketing Officer:</b> Find me every ad about <b>wireless headphones</b>. And don't show me the ones about wireless routers, I'm not an IT department.
</div>

In [14]:
q = 'body:"wireless headphones" OR headline:"wireless headphones"'
run_sql('''
    SELECT customer, category, headline, ROUND(BM25(ad_creatives, %(q)s), 3) AS bm25
    FROM ad_creatives
    WHERE BM25(ad_creatives, %(q)s)
    ORDER BY bm25 DESC
    LIMIT 8
''', {'q': q})

<div style="border-left: 4px solid #d199ff; background: rgba(209,153,255,0.10); padding: 10px 14px; margin: 8px 0;">
<b>#war-room</b> &nbsp;<span style="opacity:0.7">12:05 PM</span><br/>
<b>Morgan Blaze, Chief Marketing Officer:</b> search <b>hedphones</b> for me. i'm on my phone.
</div>

Morgan's typo is fine. `~1` allows one character of difference:

In [15]:
run_sql('''
    SELECT customer, headline
    FROM ad_creatives
    WHERE MATCH (TABLE ad_creatives) AGAINST ('body:hedphones~1')
    LIMIT 5
''')

<div class="alert alert-block alert-info">
    <b class="fa fa-solid fa-flask"></b>
    <div>
        <p><b>Evaluator's corner: BM25 vs BM25_GLOBAL</b></p>
        <p><tt>BM25</tt> scores within each partition, which is fast. <tt>BM25_GLOBAL</tt> uses statistics from every partition, which gives more consistent scores across a distributed table at some extra cost. Compare them side by side below; the ranking usually agrees, while the absolute scores differ.</p>
    </div>
</div>

In [16]:
q = 'body:(cozy rainy) OR headline:cozy'
try_sql('''
    SELECT customer, headline,
           ROUND(BM25(ad_creatives, %(q)s), 3)        AS bm25_partition,
           ROUND(BM25_GLOBAL(ad_creatives, %(q)s), 3) AS bm25_global
    FROM ad_creatives
    WHERE BM25(ad_creatives, %(q)s)
    ORDER BY bm25_global DESC
    LIMIT 8
''', {'q': q}, label='BM25_GLOBAL comparison')

## Chapter 3: "The search thinks 'running' and 'run' are different words"

<div style="border-left: 4px solid #d199ff; background: rgba(209,153,255,0.10); padding: 10px 14px; margin: 8px 0;">
<b>#war-room</b> &nbsp;<span style="opacity:0.7">2:17 PM</span><br/>
<b>Morgan Blaze, Chief Marketing Officer:</b> I searched <b>run</b> and got nothing from the running shoe people. Our biggest advertiser. Fix it before their CMO calls my CMO. (I am the CMO.)
</div>

In [17]:
run_sql('''
    SELECT COUNT(*) AS ads_matching_run
    FROM ad_creatives
    WHERE MATCH (TABLE ad_creatives) AGAINST ('body:run')
''')

### Debug the index with `ANALYZE FULLTEXT`

`ANALYZE FULLTEXT` shows exactly which tokens an analyzer produces, without touching any table. Compare the `standard` analyzer you used with the `english` one:

In [18]:
phrase = 'Running shoes for runners who ran the marathon'

def tokens(options=None):
    q = 'ANALYZE FULLTEXT %s' + (' OPTIONS %s' if options else '')
    df = try_sql(q, (phrase, options) if options else (phrase,), label='ANALYZE FULLTEXT')
    if df is None:
        return None
    return [t['token'] for t in json.loads(df.iloc[0, 0])['tokens']]

print('standard:', tokens())
print('english: ', tokens('{"analyzer": "english"}'))

The `english` analyzer drops filler words and stems words, so "Running" becomes "run". Brand names shouldn't be stemmed, though, so the new index keeps `standard` for the `customer` column with a per-column mapping.

### Swap the analyzer

A table has one full-text index, so drop it and add it back with new options. `SHOW FULLTEXT INDEX STATUS` tells you when the rebuild is done.

In [19]:
run_sql('DROP INDEX ft_ads ON ad_creatives')
run_sql('''
    ALTER TABLE ad_creatives
    ADD FULLTEXT USING VERSION 2 ft_ads (customer, headline, body)
    INDEX_OPTIONS '{"analyzer": "english", "mappings": {"customer": {"analyzer": "standard"}}}'
''')
run_sql('OPTIMIZE TABLE ad_creatives FLUSH')

for _ in range(10):
    status = try_sql('SHOW FULLTEXT INDEX STATUS FOR ad_creatives', label='Index status')
    if status is None or 'Progress (%)' not in status.columns \
            or pd.to_numeric(status['Progress (%)']).min() >= 100:
        break
    time.sleep(2)
status

In [20]:
run_sql('''
    SELECT customer, category, headline
    FROM ad_creatives
    WHERE MATCH (TABLE ad_creatives) AGAINST ('body:run')
    LIMIT 5
''')

<div style="border-left: 4px solid #d199ff; background: rgba(209,153,255,0.10); padding: 10px 14px; margin: 8px 0;">
<b>#war-room</b> &nbsp;<span style="opacity:0.7">2:31 PM</span><br/>
<b>Morgan Blaze, Chief Marketing Officer:</b> The running shoe people are happy. I am happy. You may have a snack.
</div>

## Chapter 4: "Find ads that feel like a rainy Sunday"

<div style="border-left: 4px solid #d199ff; background: rgba(209,153,255,0.10); padding: 10px 14px; margin: 8px 0;">
<b>#war-room</b> &nbsp;<span style="opacity:0.7">4:44 PM</span><br/>
<b>Morgan Blaze, Chief Marketing Officer:</b> Big idea. People don't search for products, they search for FEELINGS. Show me ads that feel like <b>a rainy Sunday</b>. No, I will not tell you what keywords that means.
</div>

Keywords can't do this, but vectors can. Every ad already has an F16 embedding. Add an HNSW vector index to make similarity search fast, then embed Morgan's feeling the same way the ads were embedded.

In [21]:
try_sql('''
    ALTER TABLE ad_creatives
    ADD VECTOR INDEX ivx (embedding)
    INDEX_OPTIONS '{"index_type": "HNSW_FLAT", "metric_type": "DOT_PRODUCT"}'
''', label='Vector index')

def set_query_vector(text):
    """Embed text into the session variable @qv, using the same model as the ads."""
    if EMBED_MODE == 'ai':
        run_sql(f'SET @qv = cluster.EMBED_TEXT(%s) :> VECTOR({DIM}) :> {VEC_TYPE}', (text,))
    else:
        v = local_model.encode([text], normalize_embeddings=True)[0]
        run_sql(f'SET @qv = %s :> {VEC_TYPE}', (json.dumps([float(x) for x in v]),))

def vibe_search(text, k=8):
    set_query_vector(text)
    return run_sql(f'''
        SELECT customer, category, headline, ROUND(embedding <*> @qv, 3) AS similarity
        FROM ad_creatives
        ORDER BY similarity DESC
        LIMIT {k}
    ''')

vibe_search('a rainy Sunday')

You should see cozy categories such as books, movies, home and games near the top, matched by meaning rather than by exact keywords. Try a few more of Morgan's moods:

In [22]:
for mood in ['getting ready for a romantic date night', 'the car is packed and the playlist is ready', 'I need to fix something this weekend']:
    print('Mood:', mood)
    display(vibe_search(mood, k=4))

<div class="alert alert-block alert-info">
    <b class="fa fa-solid fa-flask"></b>
    <div>
        <p><b>Evaluator's corner: F16 vs F32, measured on your workspace</b></p>
        <p>The next cells copy the embeddings into two plain tables, one F16 and one F32, repeat them to get a meaningful size, and compare storage, exact kNN speed and whether the two return the same top 10. Numbers depend on your workspace size and load, so run them a few times.</p>
    </div>
</div>

In [23]:
REPEAT = 20   # copies of each ad; raise for bigger tests

for t in ['bench_f16', 'bench_f32']:
    run_sql(f'DROP TABLE IF EXISTS {t}')
run_sql(f'CREATE TABLE bench_f16 (id BIGINT, embedding {VEC_TYPE}, SHARD KEY (id))')
run_sql(f'CREATE TABLE bench_f32 (id BIGINT, embedding VECTOR({DIM}), SHARD KEY (id))')

for r in range(REPEAT):
    run_sql(f'INSERT INTO bench_f16 SELECT offer_id * 1000 + {r}, embedding FROM ad_creatives WHERE embedding IS NOT NULL')
run_sql(f'INSERT INTO bench_f32 SELECT id, embedding :> VECTOR({DIM}) FROM bench_f16')
run_sql('OPTIMIZE TABLE bench_f16 FULL')
run_sql('OPTIMIZE TABLE bench_f32 FULL')

sizes = try_sql('''
    SELECT TABLE_NAME AS table_name,
           SUM(ROWS_COUNT) AS rows_stored,
           ROUND(SUM(COMPRESSED_SIZE) / 1024 / 1024, 2) AS embedding_mb_on_disk
    FROM information_schema.COLUMNAR_SEGMENTS
    WHERE DATABASE_NAME = DATABASE() AND TABLE_NAME IN ('bench_f16', 'bench_f32') AND COLUMN_NAME = 'embedding'
    GROUP BY TABLE_NAME
''', label='Storage comparison')
sizes

In [24]:
def median_ms(query, runs=7):
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        run_sql(query)
        times.append((time.perf_counter() - t0) * 1000)
    return round(statistics.median(times), 1)

moods = ['a rainy Sunday', 'date night', 'road trip', 'home workout', 'birthday gift for a kid']
overlap, t16, t32 = [], [], []
for mood in moods:
    set_query_vector(mood)
    run_sql(f'SET @qv32 = @qv :> VECTOR({DIM})')
    # speed: exact kNN over every row (no vector index on the bench tables)
    t16.append(median_ms('SELECT id FROM bench_f16 ORDER BY embedding <*> @qv DESC LIMIT 10'))
    t32.append(median_ms('SELECT id FROM bench_f32 ORDER BY embedding <*> @qv32 DESC LIMIT 10'))
    # quality: do F16 and F32 pick the same 10 distinct ads? (each ad is repeated REPEAT times)
    top = lambda t, v: set(run_sql(f'SELECT id DIV 1000 AS ad, MAX(embedding <*> {v}) AS s FROM {t} '
                               f'GROUP BY ad ORDER BY s DESC LIMIT 10')['ad'])
    overlap.append(len(top('bench_f16', '@qv') & top('bench_f32', '@qv32')) / 10)

if VEC_ELEM != 'F16':
    print('Note: F16 is not available on this workspace, so both tables below use F32. '
          'Rerun on a workspace with F16 enabled to see the real difference.')
pd.DataFrame({
    'metric': ['median exact kNN latency (ms)', 'top-10 agreement with F32'],
    f'{VEC_ELEM} (bench_f16)': [statistics.median(t16), f'{statistics.mean(overlap):.0%}'],
    'F32 (bench_f32)': [statistics.median(t32), '100% (baseline)'],
})

## Chapter 5: "Give me both. Now."

<div style="border-left: 4px solid #d199ff; background: rgba(209,153,255,0.10); padding: 10px 14px; margin: 8px 0;">
<b>#war-room</b> &nbsp;<span style="opacity:0.7">6:10 PM</span><br/>
<b>Morgan Blaze, Chief Marketing Officer:</b> Keyword search finds the exact thing. Vibe search finds the feeling. I want <b>cozy headphones for a rainy day</b> and I want BOTH kinds of smart in one answer.
</div>

**Hybrid search** runs both searches and blends the rankings with **Reciprocal Rank Fusion (RRF)**: each ad earns `1 / (60 + rank)` from each list, and the points add up. Ads that rank well in both lists float to the top. It all happens in one SQL query.

In [25]:
def lucene(text):
    """Turn free text into a safe Lucene query over the indexed columns."""
    words = ' '.join(re.findall(r'[A-Za-z0-9]+', text))
    return f'headline:({words}) OR body:({words})'

HYBRID_SQL = '''
WITH kw AS (
    SELECT offer_id, BM25(ad_creatives, %(lq)s) AS s
    FROM ad_creatives
    WHERE BM25(ad_creatives, %(lq)s)
    ORDER BY s DESC LIMIT 50
),
vec AS (
    SELECT offer_id, embedding <*> @qv AS s
    FROM ad_creatives
    ORDER BY s DESC LIMIT 50
),
ranked AS (
    SELECT offer_id, ROW_NUMBER() OVER (ORDER BY s DESC) AS r, 'keyword' AS src FROM kw
    UNION ALL
    SELECT offer_id, ROW_NUMBER() OVER (ORDER BY s DESC) AS r, 'vector' AS src FROM vec
),
fused AS (
    SELECT offer_id,
           SUM(1.0 / (60 + r)) AS rrf,
           GROUP_CONCAT(CONCAT(src, ' #', r) SEPARATOR ', ') AS found_by
    FROM ranked GROUP BY offer_id
)
SELECT a.customer, a.category, a.headline, a.max_bid_cents, ROUND(f.rrf, 4) AS rrf, f.found_by
FROM fused f JOIN ad_creatives a ON a.offer_id = f.offer_id
ORDER BY f.rrf DESC
LIMIT 10
'''

def hybrid_search(text):
    set_query_vector(text)
    return run_sql(HYBRID_SQL, {'lq': lucene(text)})

hybrid_search('cozy headphones for a rainy day')

The `found_by` column shows why each ad made the cut. Morgan can also add business rules, such as only advertisers bidding at least 10 cents, with a plain `WHERE`:

In [26]:
set_query_vector('cozy headphones for a rainy day')
run_sql(HYBRID_SQL.replace('ORDER BY f.rrf DESC', 'WHERE a.max_bid_cents >= 10\nORDER BY f.rrf DESC'),
    {'lq': lucene('cozy headphones for a rainy day')})

## Chapter 6: Super Bowl night

<div style="border-left: 4px solid #d199ff; background: rgba(209,153,255,0.10); padding: 10px 14px; margin: 8px 0;">
<b>#war-room</b> &nbsp;<span style="opacity:0.7">7:58 PM</span><br/>
<b>Morgan Blaze, Chief Marketing Officer:</b> KICKOFF IN TWO MINUTES. Advertisers are panic-buying. Expect 10x the ads. If search slows down during the halftime show I'm replacing you with a spreadsheet.
</div>

New data lands in small **segments** of the columnstore table. Each new segment gets **two separate kinds of index**, built by two different systems:

- a small **vector index** (HNSW here), built inside the SingleStore engine, and
- a small **full-text index**, built by the JLucene service (the Apache Lucene process from Chapter 2).

After a burst of writes, one query may have to check dozens of tiny indexes. **Index merging** combines per-segment indexes into bigger cross-segment ones. Vector index merging and full-text index merging solve that same problem, but they are **different mechanisms**, so this chapter handles them one at a time.

| | Vector index merging | Full-text index merging |
| --- | --- | --- |
| What gets merged | ANN graph/cluster structures (HNSW, IVF) over `embedding` | Lucene inverted indexes (term to document lists) over `customer`, `headline`, `body` |
| Built by | The SingleStore engine | The JLucene service, a separate Java process |
| How it runs | In the background (`background_vector_index_merger`), or on demand with `OPTIMIZE TABLE ... VECTOR_INDEX` | In the background with compaction steps, controlled by `enable_fts2_merger` (new, rolling out) |
| How to observe it | Segment counts in `COLUMNAR_SEGMENTS`, vector search latency | `information_schema.MV_FTS_MERGED_INDEXES`, `SHOW FULLTEXT INDEX STATUS` |
| What improves | Fewer indexes to probe per vector search | Fewer indexes to probe per keyword search, plus compaction |

Neither is the same as **columnstore segment merging** (`OPTIMIZE TABLE`), which merges the data files themselves.

Let's cause some chaos. Each "wave" adds new ads, then flushes them into their own segment:

In [27]:
def index_segments():
    """Each columnstore segment carries its own vector and full-text index,
    so counting segments shows how many small indexes a query has to probe."""
    df = try_sql('''
        SELECT COUNT(DISTINCT PARTITION, SEGMENT_ID) AS segments,
               SUM(ROWS_COUNT - DELETED_ROWS_COUNT) AS live_rows
        FROM information_schema.COLUMNAR_SEGMENTS
        WHERE DATABASE_NAME = DATABASE() AND TABLE_NAME = 'ad_creatives' AND COLUMN_NAME = 'offer_id'
    ''', label='Segment counts')
    if df is not None:
        display(df)

print('Before kickoff:')
index_segments()

In [28]:
WAVES, ADS_PER_WAVE = 12, 100   # 1,200 new ads; lower these to save AI Function quota

templates = run_sql('SELECT notification_zone, segment_ids FROM offers LIMIT 1').iloc[0]
next_id = int(run_sql('SELECT MAX(offer_id) AS m FROM offers').iloc[0, 0]) + 1
vendor_rows = vendors[['vendor', 'domain']].values.tolist()

for wave in range(1, WAVES + 1):
    rows = []
    for _ in range(ADS_PER_WAVE):
        vendor, domain = random.choice(vendor_rows)
        rows.append((next_id, vendor, True, templates['notification_zone'], json.dumps([]),
                     f'{random.randint(10, 70)}% off at {vendor} during the big game', domain,
                     random.randint(5, 40)))
        next_id += 1
    with conn.cursor() as cur:
        cur.executemany('INSERT INTO offers VALUES (%s, %s, %s, %s, %s, %s, %s, %s)', rows)
    run_sql('CALL build_creatives()')
    embed_pending()
    run_sql('OPTIMIZE TABLE ad_creatives FLUSH')   # each wave becomes its own segment
    print(f'Wave {wave}: {ADS_PER_WAVE} ads live')

print('\nAfter the rush:')
index_segments()

Two stopwatches, one per index type, so each merge is measured on its own:

In [29]:
QUERY = 'snacks and a big screen for the game'
set_query_vector(QUERY)
LQ = lucene(QUERY)

VECTOR_ONLY = 'SELECT offer_id FROM ad_creatives ORDER BY embedding <*> @qv DESC LIMIT 10'
KEYWORD_ONLY = '''
    SELECT offer_id, customer, BM25(ad_creatives, %(lq)s) AS s
    FROM ad_creatives WHERE BM25(ad_creatives, %(lq)s)
    ORDER BY s DESC LIMIT 10
'''

def latency_ms(query, params=None, runs=9):
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        run_sql(query, params)
        times.append((time.perf_counter() - t0) * 1000)
    return round(statistics.median(times), 1)

results = {'vector search (ms)': [latency_ms(VECTOR_ONLY)],
           'keyword search (ms)': [latency_ms(KEYWORD_ONLY, {'lq': LQ})]}
top_keyword_before = run_sql(KEYWORD_ONLY, {'lq': LQ})
pd.DataFrame(results, index=['after the rush'])

### Halftime, part 1: vector index merging

The vector index merger combines the per-segment HNSW indexes into a cross-segment index, so a vector search probes a few large graphs instead of many small ones. It runs on its own, but you can trigger it:

In [30]:
try_sql('OPTIMIZE TABLE ad_creatives VECTOR_INDEX ivx', label='Vector index merge')
results['vector search (ms)'].append(latency_ms(VECTOR_ONLY))
print(f"Vector search: {results['vector search (ms)'][0]} ms -> {results['vector search (ms)'][1]} ms")

### Halftime, part 2: full-text index merging (new, rolling out)

This is a different machine entirely. The full-text indexes are Lucene indexes managed by the JLucene service. The **full-text index merger** is a brand-new feature that combines per-segment Lucene indexes into cross-segment indexes and compacts them, in the background, so a keyword search probes a few large indexes instead of many small ones.

It's still rolling out, so it may not be active on your workspace yet. It has its own switch, `enable_fts2_merger`, separate from the vector merger's `background_vector_index_merger`. When it's active, its work shows up in `information_schema.MV_FTS_MERGED_INDEXES`: one row per merged index, with how many segments (`NUM_SEGMENTS`) and rows (`NUM_ROWS`) it combines and its `SIZE`. The next cell shows both switches and whatever the merger has produced so far.

In [31]:
settings = run_sql("SHOW VARIABLES WHERE Variable_name IN ('enable_fts2_merger', 'background_vector_index_merger')")
display(settings)

merged = try_sql('''
    SELECT PARTITION, FTS_INDEX_ID, NUM_SEGMENTS, NUM_ROWS, SIZE
    FROM information_schema.MV_FTS_MERGED_INDEXES
    WHERE DATABASE_NAME = DATABASE() AND TABLE_NAME = 'ad_creatives'
    ORDER BY PARTITION
''', label='MV_FTS_MERGED_INDEXES')

if merged is not None and len(merged) > 0:
    print(f'The full-text merger combined per-segment indexes into {len(merged)} merged index(es):')
    display(merged)
else:
    print('No merged full-text indexes on this workspace yet. The full-text index merger is new and still rolling out; '
          'once it is active, this view fills in automatically as new segments arrive. Nothing to change in the notebook.')

status = try_sql('SHOW FULLTEXT INDEX STATUS FOR ad_creatives', label='SHOW FULLTEXT INDEX STATUS')
if status is not None:
    display(status)

results['keyword search (ms)'].append(latency_ms(KEYWORD_ONLY, {'lq': LQ}))
print(f"Keyword search: {results['keyword search (ms)'][0]} ms -> {results['keyword search (ms)'][1]} ms")
print('(Background merges of data segments and warm caches also affect this number, so compare several runs.)')

Speed is only half the story. A merge should never change **what** you find: the same ads, in the same order, with the same BM25 scores. Check that the top keyword results match before and after:

In [32]:
top_keyword_after = run_sql(KEYWORD_ONLY, {'lq': LQ})
pd.DataFrame({
    'before: advertiser': top_keyword_before['customer'].values,
    'before: bm25': top_keyword_before['s'].round(3).values,
    'after: advertiser': top_keyword_after['customer'].values,
    'after: bm25': top_keyword_after['s'].round(3).values,
})

Finally, a regular columnstore merge tidies up the data segments themselves, and a scoreboard for Morgan:

In [33]:
try_sql('OPTIMIZE TABLE ad_creatives', label='Columnstore merge')
index_segments()
pd.DataFrame(results, index=['after the rush', 'after merging'])

<div class="alert alert-block alert-info">
    <b class="fa fa-solid fa-flask"></b>
    <div>
        <p><b>Evaluator's corner: what to expect from merging</b></p>
        <p>With a few thousand rows the difference can be small, because every index is tiny. The effect grows with data volume and write rate: many small segments mean many index lookups per query, and merging turns them into a few large ones. Both mergers may also have run on their own during the waves, which is the point: they keep up in the background. For a fair test, load a larger dataset and measure vector and keyword latency separately during a sustained write load.</p>
    </div>
</div>

<div style="border-left: 4px solid #d199ff; background: rgba(209,153,255,0.10); padding: 10px 14px; margin: 8px 0;">
<b>#war-room</b> &nbsp;<span style="opacity:0.7">11:59 PM</span><br/>
<b>Morgan Blaze, Chief Marketing Officer:</b> Search never blinked. Advertisers are thrilled. I'm telling the board you're a genius. Also, tomorrow I want it in Spanish.
</div>

(Good news for tomorrow: the full-text index supports a `spanish` analyzer. Swapping it in is the same `DROP INDEX` / `ADD FULLTEXT` move from Chapter 3.)

## What you built

| Feature | Where you used it |
| --- | --- |
| Pipelines + AI Functions | `purchases_pipeline` streamed real purchase events and created ads and embeddings with `cluster.EMBED_TEXT` while loading |
| F16 vectors | `VECTOR(<dims>, F16)` column, about half the storage of F32 |
| Full-text search VERSION 2 | Phrases, fuzzy matching, `BM25` and `BM25_GLOBAL` |
| Full-text manageability | `ALTER TABLE ... ADD FULLTEXT` on live data, `ANALYZE FULLTEXT`, `SHOW FULLTEXT INDEX STATUS`, per-column analyzers |
| Hybrid search | Keyword + vector blended with Reciprocal Rank Fusion in one query |
| Vector index merging | `OPTIMIZE TABLE ... VECTOR_INDEX` combined per-segment HNSW indexes |
| Full-text index merging (rolling out) | Where the new merger lives, how it differs from vector merging, and how to watch it in `MV_FTS_MERGED_INDEXES` |

### Where to take it next

- **Developers:** point your app at `hybrid_search()`'s SQL, and leave `purchases_pipeline` running without `FOREGROUND` (and raise `SCALE`) so new advertisers are searchable as they appear.
- **Evaluators:** raise `REPEAT` and `WAVES` to test at your own scale, and compare the F16 numbers against your current vector store.
- **Marketing teams:** the same pattern powers product search, content recommendations and support-ticket search. Swap the dataset, keep the SQL.

## Clean up

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Action Required</b></p>
        <p>If you created a new database in your Standard or Premium Workspace, you can drop it by running the cell below. On a Free Starter Workspace this drops only the tables and pipeline this notebook made.</p>
    </div>
</div>

In [34]:
try_sql('DROP PIPELINE IF EXISTS purchases_pipeline', label='Drop pipeline')
if not IS_SHARED:
    run_sql(f'DROP DATABASE IF EXISTS {DB_NAME}')
else:
    for t in ['bench_f16', 'bench_f32', 'ad_creatives', 'creatives_staging', 'offers', 'purchases', 'category_vibes', 'vendors']:
        run_sql(f'DROP TABLE IF EXISTS {t}')
    for p in ['ingest_purchases', 'refresh_ads', 'build_creatives']:
        try_sql(f'DROP PROCEDURE IF EXISTS {p}', label=f'Drop {p}')
print('All clean. Morgan will never know.')

<div id="singlestore-footer" style="background-color: rgba(194, 193, 199, 0.25); height:2px; margin-bottom:10px"></div>
<div><img src="https://raw.githubusercontent.com/singlestore-labs/spaces-notebooks/master/common/images/singlestore-logo-grey.png" style="padding: 0px; margin: 0px; height: 24px"/></div>